# TAT Analysis Dashboard

Click the button below to run the analysis and display the charts. You can upload a different CSV file if desired, or the default file (`ConsolidatedTATReportNew.csv`) from the repository will be used.

<button onclick="Jupyter.notebook.execute_all_cells();">Run Analysis</button>


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import interact
import os
import io

    # Function to classify shifts based on time of day (24-hour operation)
def classify_shift(time):
    if pd.isna(time):
        return None
    if time.hour >= 20 or time.hour < 7:
        return 'Night Shift'
    elif 7 <= time.hour < 10:
        return 'Morning'
    elif 10 <= time.hour < 13:
        return 'Mid Morning'
    elif 13 <= time.hour < 16:
        return 'Afternoon'
    elif 16 <= time.hour < 20:
        return 'Evening'
    return None

   # Main data processing function to load and process CSV data
def fetch_and_process_data(file_content=None, file_path=None):
    # If file_path is provided, read from the file path (used for repo CSV)
    if file_path:
        try:
            TAT_df = pd.read_csv(file_path, dtype={'UHID': str})
        except Exception as e:
            print(f"Error reading the CSV file from repository: {e}")
            return None
    # If file_content is provided, read from the uploaded file (used for widget)
    elif file_content:
        try:
            TAT_df = pd.read_csv(io.BytesIO(file_content), dtype={'UHID': str})
        except Exception as e:
            print(f"Error reading the uploaded CSV file: {e}")
            return None
    else:
        print("No file content or path provided.")
        return None

    # Define datetime columns to parse
    datetime_columns = [
        'ConsultationBillingTime',
        'Pharmacy_Billing_Time'
    ]

    # Print raw column names for debugging
    print("\nRaw column names from CSV:", TAT_df.columns.tolist())

    # Inspect raw datetime columns
    print("\nRaw datetime columns dtypes and sample values:")
    for col in datetime_columns:
        if col in TAT_df.columns:
            print(f"{col}: dtype={TAT_df[col].dtype}")
            print(f"Sample values:\n{TAT_df[col].head()}\n")
            print(f"Value counts:\n{TAT_df[col].value_counts(dropna=False).head()}\n")

    # Convert datetime columns
    for col in datetime_columns:
        if col in TAT_df.columns:
            print(f"Converting {col} to datetime...")
            TAT_df[col] = pd.to_datetime(TAT_df[col], dayfirst=True, errors='coerce')

    # Verify datetime parsing
    print("\nDatetime columns dtypes after conversion:")
    for col in datetime_columns:
        if col in TAT_df.columns:
            print(f"{col}: dtype={TAT_df[col].dtype}")
            print(f"Sample values after conversion:\n{TAT_df[col].head()}\n")

    # Select required columns for TAT calculation
    columns_to_import = [
        'UHID', 'PatientName', 'Department', 'FacilityName',
        'ConsultationBillingTime', 'Pharmacy_Billing_Time'
    ]

    # Check for missing columns
    missing_cols = [col for col in columns_to_import if col not in TAT_df.columns]
    if missing_cols:
        print(f"Error: Missing required columns in CSV: {missing_cols}")
        return None
    
    # Keep only the required columns
    TAT_df = TAT_df[columns_to_import].copy()

    # Filter out invalid data
    filtered_TAT_df = TAT_df.dropna(subset=['UHID'])
    filtered_TAT_df = filtered_TAT_df[filtered_TAT_df['FacilityName'] != "Bliss Medical Centre HomeCare"]

    # Split data by department
    Consultation_df = filtered_TAT_df[filtered_TAT_df['Department'] == 'GENERAL OPD'].drop(
        columns=['Pharmacy_Billing_Time'] if 'Pharmacy_Billing_Time' in filtered_TAT_df.columns else []).copy()
    Pharmacy_df = filtered_TAT_df[filtered_TAT_df['Department'] == 'Pharmacy'].drop(
        columns=['ConsultationBillingTime'] if 'ConsultationBillingTime' in filtered_TAT_df.columns else []).copy()

    # Add date columns for grouping
    Consultation_df['date'] = Consultation_df['ConsultationBillingTime'].dt.date
    Pharmacy_df['date'] = Pharmacy_df['Pharmacy_Billing_Time'].dt.date

    # Group and aggregate data by taking the earliest timestamps
    TAT_pharmacy_df = Pharmacy_df.groupby(['date', 'UHID', 'PatientName', 'FacilityName']).agg({
        'Pharmacy_Billing_Time': 'min',
        'Department': 'first'
    }).reset_index()

    TAT_consultation_df = Consultation_df.groupby(['date', 'UHID', 'PatientName', 'FacilityName']).agg({
        'ConsultationBillingTime': 'min',
        'Department': 'first'
    }).reset_index()

    # Create unique identifiers for matching records
    for df in [TAT_pharmacy_df, TAT_consultation_df]:
        df['Unique'] = df['UHID'].astype(str) + "_" + \
                        df['PatientName'].astype(str) + "_" + \
                        df['FacilityName'].astype(str) + "_" + \
                        df['date'].astype(str)

    # Merge consultation and pharmacy data
    merged_df = TAT_consultation_df.merge(
        TAT_pharmacy_df[['Unique', 'Pharmacy_Billing_Time']],
        on='Unique',
        how='left'
    )

    # Calculate TAT for matched records
    filtered_merged_df = merged_df[merged_df['Pharmacy_Billing_Time'].notna()].copy()
    filtered_merged_df['TAT'] = (filtered_merged_df['Pharmacy_Billing_Time'] - 
                                filtered_merged_df['ConsultationBillingTime']).dt.total_seconds() / 60
    # Remove records with negative TAT by filtering
    filtered_merged_df = filtered_merged_df[filtered_merged_df['TAT'] >= 0].copy()
    filtered_merged_df['Time_out'] = filtered_merged_df['Pharmacy_Billing_Time']
    filtered_merged_df['Department'] = 'TAT'

    # No time filter needed since we want 24-hour data
    filtered_period_df = filtered_merged_df.copy()

    if filtered_period_df.empty:
        print("No data available after processing.")
        return None

    # Calculate minutes since midnight for plotting (24-hour range)
    filtered_period_df['Minutes_Since_Midnight'] = (
        filtered_period_df['Time_out'].dt.hour * 60 +
        filtered_period_df['Time_out'].dt.minute
    )

    # Extract hour for hourly table
    filtered_period_df['Hour'] = filtered_period_df['Time_out'].dt.hour

    # Add shift and date components
    filtered_period_df['Shift'] = filtered_period_df['Time_out'].apply(classify_shift)
    filtered_period_df['Year'] = filtered_period_df['Time_out'].dt.year
    filtered_period_df['Month'] = filtered_period_df['Time_out'].dt.month
    filtered_period_df['Day'] = filtered_period_df['Time_out'].dt.day
    filtered_period_df['Date'] = pd.to_datetime(filtered_period_df[['Year', 'Month', 'Day']])
    
    # Add Time column by extracting the time portion from Time_out
    filtered_period_df['Time'] = filtered_period_df['Time_out'].dt.time

    print(f"\nTAT records: {len(filtered_period_df)}")

    # Display filtered data
    print("\nFiltered Data (24 Hours):")
    display(filtered_period_df[['PatientName', 'FacilityName', 'Time_out', 'Time', 'TAT', 'Minutes_Since_Midnight', 'Shift', 'Year', 'Month', 'Day', 'Unique']])

    # Print unique filter values for debugging
    print("\nUnique filter values:")
    print("Years:", filtered_period_df['Year'].unique())
    print("Months:", filtered_period_df['Month'].unique())
    print("Days:", filtered_period_df['Day'].unique())
    print("Facilities:", filtered_period_df['FacilityName'].unique())

    return filtered_period_df

# Function to create a plot with TAT trend and a table below it
def plot_tat_trend(df, start_year, start_month, start_day, end_year, end_month, end_day, facility, output_dir, save=True):
    # Construct start and end dates
    start_date = pd.to_datetime(f"{start_year}-{start_month}-{start_day}")
    end_date = pd.to_datetime(f"{end_year}-{end_month}-{end_day}")

    # Ensure start_date <= end_date
    if start_date > end_date:
        start_date, end_date = end_date, start_date
        print(f"Swapped dates: Start date {start_date}, End date {end_date}")

    # Filter data based on the date range
    filtered_df = df[
        (df['Date'] >= start_date) &
        (df['Date'] <= end_date)
    ]

    if filtered_df.empty:
        print(f"No data available for the selected date range: {start_date.date()} to {end_date.date()}.")
        # Fallback: Use the full date range
        earliest_date = df['Date'].min()
        latest_date = df['Date'].max()
        print(f"Falling back to full date range: {earliest_date.date()} to {latest_date.date()}")
        filtered_df = df[
            (df['Date'] >= earliest_date) &
            (df['Date'] <= latest_date)
        ]
        start_date, end_date = earliest_date, latest_date

    if filtered_df.empty:
        print("No data available even with fallback filters.")
        return

    # If a specific facility is selected (not "All Facilities"), filter by facility
    if facility != "All Facilities":
        filtered_df = filtered_df[filtered_df['FacilityName'] == facility]
        if filtered_df.empty:
            print(f"No data available for facility: {facility} in the selected date range.")
            return

    # Create figure with two subplots: one for the graph, one for the table
    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.3)

    # --- Plot the TAT Trend (Top Subplot) ---
    ax1 = fig.add_subplot(gs[0, 0])

    # Plot TAT Trend (per minute, left y-axis)
    minutes = filtered_df['Minutes_Since_Midnight']
    tat = filtered_df['TAT']

    # Create a full minute range (00:00 to 23:59 = 1440 minutes)
    all_minutes = np.arange(0, 1440)  # 0 to 1439 minutes (24 hours)
    tat_full = np.full_like(all_minutes, np.nan, dtype=float)  # Initialize with NaN

    # Group by minute and average TAT across all days (and facilities if "All Facilities")
    minute_groups = filtered_df.groupby('Minutes_Since_Midnight')['TAT'].mean()
    for min_val, tat_val in minute_groups.items():
        tat_full[int(min_val)] = tat_val

    # Interpolate to fill gaps (linear interpolation)
    tat_series = pd.Series(tat_full)
    tat_interpolated = tat_series.interpolate(method='linear')

    # Plot TAT on the left y-axis
    label = 'Overall TAT (All Facilities)' if facility == "All Facilities" else f'{facility} TAT'
    ax1.plot(all_minutes, tat_interpolated, linewidth=2, label=label, color='blue')
    ax1.set_xlabel('Time of Day', fontsize=12)
    ax1.set_ylabel('Average TAT (Minutes)', fontsize=12, color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    # Set x-axis with 1-hour intervals (00:00 to 23:00)
    title = f'Overall TAT Trend (24 Hours) - All Facilities - {start_date.date()} to {end_date.date()}' if facility == "All Facilities" else f'TAT Trend (24 Hours) - {facility} - {start_date.date()} to {end_date.date()}'
    ax1.set_title(title, fontsize=14)
    ax1.set_xticks(np.arange(0, 1440, 60))
    ax1.set_xticklabels([f'{h:02d}:00' for h in range(0, 24)], rotation=45)
    ax1.grid(True, alpha=0.3)

    # Add legend
    ax1.legend(loc='upper left')

    # --- Create the Hourly Table (Bottom Subplot) ---
    ax_table = fig.add_subplot(gs[1, 0])

    # Group by hour to calculate average TAT and count of unique records (footfalls)
    hourly_stats = filtered_df.groupby('Hour').agg({
        'TAT': 'mean',           # Average TAT per hour
        'Unique': 'nunique'      # Count of unique records (footfalls) per hour
    }).reindex(range(24), fill_value=0)  # Ensure all hours (0-23) are present

    # Round average TAT to 2 decimal places
    hourly_stats['TAT'] = hourly_stats['TAT'].round(2)

    # Prepare table data
    table_data = [
        hourly_stats['TAT'].values,      # Row 1: Average TAT
        hourly_stats['Unique'].values    # Row 2: Count of Unique (Footfalls)
    ]
    row_labels = ['Avg TAT (min)', 'Footfalls']
    col_labels = [f'{h}' for h in range(24)]  # Column labels: 0, 1, ..., 23

    # Create the table
    table = ax_table.table(cellText=table_data,
                            rowLabels=row_labels,
                            colLabels=col_labels,
                            cellLoc='center',
                            loc='center')
    
    # Style the table
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.5)  # Adjust table size
    ax_table.axis('off')  # Hide the axes for the table

    # Prepare CSV export data
    stats_df = filtered_df.groupby(['FacilityName', 'Hour']).agg({
        'TAT': 'mean',           # Average TAT per hour per facility
        'Unique': 'nunique'      # Count of unique records (footfalls) per hour per facility
    }).reset_index()

    # Round average TAT to 2 decimal places
    stats_df['TAT'] = stats_df['TAT'].round(2)

    # Ensure all hours (0-23) are present for each facility
    facilities = filtered_df['FacilityName'].unique()
    all_hours = range(24)
    all_combinations = pd.MultiIndex.from_product([facilities, all_hours], names=['FacilityName', 'Hour'])
    all_combinations_df = pd.DataFrame(index=all_combinations).reset_index()
    
    # Merge with stats_df to fill in missing hours with 0
    stats_df = all_combinations_df.merge(stats_df, on=['FacilityName', 'Hour'], how='left')
    stats_df['TAT'] = stats_df['TAT'].fillna(0)
    stats_df['Unique'] = stats_df['Unique'].fillna(0)

    # Format the Hours column as "Xam" or "Xpm"
    stats_df['Hours'] = stats_df['Hour'].apply(lambda x: f"{x % 12 if x % 12 != 0 else 12}{'am' if x < 12 else 'pm'}")

    # Prepare CSV data
    csv_data = stats_df[['FacilityName', 'Hours', 'TAT', 'Unique']].copy()
    csv_data.rename(columns={'Unique': 'Footfalls'}, inplace=True)

    # Export table data to CSV in the current directory
    csv_filename = f"tat_stats_{start_date.date()}_to_{end_date.date()}.csv"
    csv_data.to_csv(csv_filename, index=False)
    print(f"Table data exported to {csv_filename}. Download it from the Jupyter file browser.")

    # Adjust layout to prevent overlap
    plt.tight_layout()

    if save:
        filename = 'tat_trend_all_facilities.png' if facility == "All Facilities" else 'tat_trend.png'
        plt.savefig(filename)
    plt.show()
    plt.close()

# Interactive plot function with dynamic date range and facility filter
def interactive_plot(df):
    if df is None or df.empty:
        print("No data available after processing.")
        return

    # Get unique values for filters
    years = sorted(df['Year'].dropna().unique().astype(int))
    if not years:
        print("No valid years available in the data.")
        return

    # Create facility options with "All Facilities" as the default
    facility_options = ['All Facilities'] + sorted(df['FacilityName'].unique().tolist())
    if len(facility_options) <= 1:  # Only "All Facilities" (no actual facilities)
        print("No facilities available in the data.")
        return

    # Create widgets for start date
    start_year_widget = widgets.Dropdown(options=years, description='Start Year:', value=years[0])
    start_month_widget = widgets.Dropdown(options=[1], description='Start Month:', value=1)
    start_day_widget = widgets.Dropdown(options=[1], description='Start Day:', value=1)

    # Create widgets for end date
    end_year_widget = widgets.Dropdown(options=years, description='End Year:', value=years[-1])
    end_month_widget = widgets.Dropdown(options=[1], description='End Month:', value=1)
    end_day_widget = widgets.Dropdown(options=[1], description='End Day:', value=1)

    # Facility widget with "All Facilities" as default
    facility_widget = widgets.Dropdown(options=facility_options, description='Facility:', value='All Facilities')

    # Update start month options based on start year
    def update_start_month_options(*args):
        selected_year = start_year_widget.value
        valid_months = sorted(df[df['Year'] == selected_year]['Month'].dropna().unique().astype(int))
        start_month_widget.options = valid_months if valid_months else [1]
        start_month_widget.value = valid_months[0] if valid_months else 1
        update_start_day_options()

    # Update start day options based on start year and month
    def update_start_day_options(*args):
        selected_year = start_year_widget.value
        selected_month = start_month_widget.value
        valid_days = sorted(df[(df['Year'] == selected_year) & (df['Month'] == selected_month)]['Day'].dropna().unique().astype(int))
        start_day_widget.options = valid_days if valid_days else [1]
        start_day_widget.value = valid_days[0] if valid_days else 1

    # Update end month options based on end year
    def update_end_month_options(*args):
        selected_year = end_year_widget.value
        valid_months = sorted(df[df['Year'] == selected_year]['Month'].dropna().unique().astype(int))
        end_month_widget.options = valid_months if valid_months else [1]
        end_month_widget.value = valid_months[-1] if valid_months else 1
        update_end_day_options()

    # Update end day options based on end year and month
    def update_end_day_options(*args):
        selected_year = end_year_widget.value
        selected_month = end_month_widget.value
        valid_days = sorted(df[(df['Year'] == selected_year) & (df['Month'] == selected_month)]['Day'].dropna().unique().astype(int))
        end_day_widget.options = valid_days if valid_days else [1]
        end_day_widget.value = valid_days[-1] if valid_days else 1

    # Attach observers for dynamic updates
    start_year_widget.observe(update_start_month_options, 'value')
    start_month_widget.observe(update_start_day_options, 'value')
    end_year_widget.observe(update_end_month_options, 'value')
    end_month_widget.observe(update_end_day_options, 'value')

    # Initialize month and day options
    update_start_month_options()
    update_end_month_options()

    # Define interactive function with date range and facility filter
    @interact(
        start_year=start_year_widget, start_month=start_month_widget, start_day=start_day_widget,
        end_year=end_year_widget, end_month=end_month_widget, end_day=end_day_widget,
        facility=facility_widget
    )
    def update_plot(start_year, start_month, start_day, end_year, end_month, end_day, facility):
        plot_tat_trend(df, start_year, start_month, start_day, end_year, end_month, end_day, facility, output_dir='')

# Create a file upload widget for optional user uploads
upload_widget = widgets.FileUpload(
    accept='.csv',  # Accept only CSV files
    multiple=False  # Allow only one file to be uploaded
)

# Display the upload widget
display(upload_widget)

# Function to process the uploaded file (or repo file) and run the interactive plot
def on_upload_change(change):
    if upload_widget.value:
        # User uploaded a file, process it
        file_content = next(iter(upload_widget.value.values()))['content']
        df = fetch_and_process_data(file_content=file_content)
        print("Using uploaded CSV file.")
    else:
        # No file uploaded, use the CSV file from the repository
        csv_path = 'data/ConsolidatedTATReportNew.csv'  # Update this path if the CSV is in a subdirectory (e.g., 'data/ConsolidatedTATReportNew.csv')
        print(f"Using default CSV file from repository: {csv_path}")
        df = fetch_and_process_data(file_path=csv_path)

    if df is not None:
        try:
            interactive_plot(df)
        except Exception as e:
            print(f"Error with interactive plot: {e}")
            if not df.empty:
                earliest_date = df['Date'].min()
                latest_date = df['Date'].max()
                print(f"Generating static plot for {earliest_date.date()} to {latest_date.date()}")
                plot_tat_trend(
                    df,
                    earliest_date.year, earliest_date.month, earliest_date.day,
                    latest_date.year, latest_date.month, latest_date.day,
                    "All Facilities",
                    output_dir=''  # Use current directory
                )
            else:
                print("No data available for plotting.")
    else:
        print("No data available for plotting.")

# Observe changes to the upload widget
upload_widget.observe(on_upload_change, names='value')

# Automatically trigger the function to load the repo CSV by default
on_upload_change(None)

FileUpload(value=(), accept='.csv', description='Upload')

Using default CSV file from repository: data/ConsolidatedTATReportNew.csv


/tmp/ipykernel_100665/1189378015.py:31: DtypeWarning: Columns (8,9,10,11,12,14,16,17,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  TAT_df = pd.read_csv(file_path, dtype={'UHID': str})



Raw column names from CSV: ['UHID', 'PatientName', 'Department', 'DoctorName', 'FacilityName', 'Service', 'ItemName', 'AppointmentTime', 'ConsultationBillingTime', 'Vitals_Captured_Date_Time', 'ConsultationStart_Date_Time', 'Visit_Signed_Time', 'Visit_Modify_Date_Time', 'Service_Bill_Date_Time', 'Procedure_Completion_Date_', 'Sample_Collection__Acknowledge_Date___Time', 'ID_Report_Save_Date_and_Time', 'Report_Provisional_Date_and_Time', 'Report_verify_Date_and_Time', 'Pharmacy_Billing_Time', 'Pharmacy_Dispensing_Time', 'Refund_Return_Date_time_']

Raw datetime columns dtypes and sample values:
ConsultationBillingTime: dtype=object
Sample values:
0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Name: ConsultationBillingTime, dtype: object

Value counts:
ConsultationBillingTime
NaN                 195456
07/04/2025 10:56        23
07/04/2025 11:19        23
17/04/2025 11:54        23
15/04/2025 11:44        22
Name: count, dtype: int64

Pharmacy_Billing_Time: dtype=object
Sample values:
0  

,PatientName,FacilityName,Time_out,Time,TAT,Minutes_Since_Midnight,Shift,Year,Month,Day,Unique
0,Mast. IAN KIPROP,Bliss Medical Centre Kabarnet,2025-04-01 10:16:00,10:16:00,21.0,616,Mid Morning,2025,4,1,BM01.10346_Mast. IAN KIPROP_Bliss Medical Cen...
1,Ms. CAROLYNE CHEBOR,Bliss Medical Centre Kabarnet,2025-04-01 17:25:00,17:25:00,27.0,1045,Evening,2025,4,1,BM01.10463_Ms. CAROLYNE CHEBOR_Bliss Medical ...
2,Mr. ROBINSON CHELAL,Bliss Medical Centre Zion Mall,2025-04-01 16:55:00,16:55:00,94.0,1015,Evening,2025,4,1,BM01.10804_Mr. ROBINSON CHELAL_Bliss Medical ...
3,Mast. BAVINE JEPKOECH,Bliss Medical Centre Kabarnet,2025-04-01 17:08:00,17:08:00,114.0,1028,Evening,2025,4,1,BM01.11630_Mast. BAVINE JEPKOECH_Bliss Medica...
4,Mrs. SALOME JEMATIA,Bliss Medical Centre Kabarnet,2025-04-01 18:18:00,18:18:00,24.0,1098,Evening,2025,4,1,BM01.12527_Mrs. SALOME JEMATIA_Bliss Medical ...
...,...,...,...,...,...,...,...,...,...,...,...
56690,Mrs. JACKLINE NGIGI SIGEI,Medicross Medical Centre Nakuru,2025-04-23 15:16:00,15:16:00,69.0,916,Afternoon,2025,4,23,BM74.588_Mrs. JACKLINE NGIGI SIGEI_Medicross M...
56692,Mrs. JOSEPHINE NTHENYA,Medicross Medical Centre Nakuru,2025-04-23 11:34:00,11:34:00,87.0,694,Mid Morning,2025,4,23,BM74.794_Mrs. JOSEPHINE NTHENYA_Medicross Med...
56693,Ms. IFTIZAM ABDI,Medicross Medical Centre Nakuru,2025-04-23 13:02:00,13:02:00,90.0,782,Afternoon,2025,4,23,BM74.8076_Ms. IFTIZAM ABDI _Medicross Medical...
56694,Mr. DANIEL LESANGALE TAPARKWE,Medicross Medical Centre Nakuru,2025-04-23 14:57:00,14:57:00,75.0,897,Afternoon,2025,4,23,BM74.8934_Mr. DANIEL LESANGALE TAPARKWE_Medic...



Unique filter values:
Years: [2025]
Months: [4]
Days: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
Facilities: ['Bliss Medical Centre Kabarnet' 'Bliss Medical Centre Zion Mall'
 'Bliss Medical Centre Iten' 'Bliss Medical Centre Eldama Ravine'
 'Bliss Medical Centre Ol Kalau' 'Bliss Medical Centre Webuye'
 'Bliss Medical Centre Moi Avenue' 'Bliss Medical Centre Kimilili'
 'Bliss Medical Centre Kitale' 'Bliss Medical Centre Nanyuki'
 'Bliss Medical Centre Pipeline' 'Bliss Medical Centre Kerugoya'
 'Bliss MC Westlands' 'Bliss Medical Centre Busia'
 'Bliss Medical Centre Siaya' 'Bliss Medical Centre Machakos'
 'Bliss Medical Centre Kakamega' 'Bliss Medical Centre Embu'
 'Bliss Medical Centre Muranga' 'Medicross Medical Centre Juja Mall'
 'Medicross Medical Centre Meru'
 'Bliss Medical Centre Nyeri 1 Jenkim Plaza' 'BLISS MC EASTLEIGH'
 'Bliss Medical Centre Kiambu' 'Bliss Medical Centre Homabay'
 'Bliss Medical Centre Kisii 1' 'Bliss Medical Centre Al Imran'
 'Bli

interactive(children=(Dropdown(description='Start Year:', options=(np.int64(2025),), value=np.int64(2025)), Dr…